In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest,chi2,RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier   
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier



def rfeFeature(indep_X, dep_Y,n):

    log_model = LogisticRegression(solver='lbfgs',max_iter=20000)
    RF = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
    DT= DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
    svc_model = SVC(kernel = 'linear', random_state = 0)
    
    rfelist=[]
    rfemodellist=[log_model,svc_model,RF,DT]
    
    for i in rfemodellist:
        log_rfe=RFE(i, n_features_to_select=n, step=1)
        log_fit = log_rfe.fit(indep_X, dep_Y)
        log_rfe_feature=log_fit.transform(indep_X)
        rfelist.append(log_rfe_feature)
    return rfelist

def split_scaler(indep_X,dep_Y):
    X_train,X_test,y_train,y_test=train_test_split(indep_X,dep_Y,test_size=0.25,random_state=0)
    sc=StandardScaler()
    X_train=sc.fit_transform(X_train)
    X_test=sc.transform(X_test)
    return X_train,X_test,y_train,y_test

def cm_predictions(classifier,X_test,y_test):
    y_pred=classifier.predict(X_test)
    from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
    cm=confusion_matrix(y_test,y_pred)
    accuracy=accuracy_score(y_test,y_pred)
    report=classification_report(y_test,y_pred)
    return classifier,accuracy,report,X_test,y_test,cm

def logistic(X_train,X_test,y_train,y_test):
    from sklearn.linear_model import LogisticRegression
    classifier=LogisticRegression(random_state=0)
    classifier.fit(X_train,y_train)
    classifier,accuracy,report,X_test,y_test,cm=cm_predictions(classifier,X_test,y_test)
    return classifier,accuracy,report,X_test,y_test,cm

def svm_linear(X_train,X_test,y_train,y_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel = 'linear', random_state = 0)
    classifier.fit(X_train,y_train)
    classifier,accuracy,report,X_test,y_test,cm=cm_predictions(classifier,X_test,y_test)
    return classifier,accuracy,report,X_test,y_test,cm

def svm_NL(X_train,X_test,y_train,y_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel = 'rbf', random_state = 0)
    classifier.fit(X_train,y_train)
    classifier,accuracy,report,X_test,y_test,cm=cm_predictions(classifier,X_test,y_test)
    return classifier,accuracy,report,X_test,y_test,cm

def Navie(X_train,X_test,y_train,y_test):
    from sklearn.naive_bayes import GaussianNB
    classifier = GaussianNB()
    classifier.fit(X_train,y_train)
    classifier,accuracy,report,X_test,y_test,cm=cm_predictions(classifier,X_test,y_test)
    return classifier,accuracy,report,X_test,y_test,cm

def knn(X_train,X_test,y_train,y_test):
    from sklearn.neighbors import KNeighborsClassifier
    classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
    classifier.fit(X_train,y_train)
    classifier,accuracy,report,X_test,y_test,cm=cm_predictions(classifier,X_test,y_test)
    return classifier,accuracy,report,X_test,y_test,cm
    
def decision(X_train,X_test,y_train,y_test):
    from sklearn.tree import DecisionTreeClassifier
    classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
    classifier.fit(X_train,y_train)
    classifier,accuracy,report,X_test,y_test,cm=cm_predictions(classifier,X_test,y_test)
    return classifier,accuracy,report,X_test,y_test,cm

def random(X_train,X_test,y_train,y_test):
    from sklearn.ensemble import RandomForestClassifier
    classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
    classifier.fit(X_train,y_train)
    classifier,accuracy,report,X_test,y_test,cm=cm_predictions(classifier,X_test,y_test)
    return classifier,accuracy,report,X_test,y_test,cm

def rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf):
    dataframe=pd.DataFrame({'Logistic':acclog,
                            'SVMl':accsvml,
                            'SVMnl':accsvmnl,
                            'KNN': accknn,
                            'Navie':accnav,
                            'Decision':accdes,
                            'Random':accrf} , index =['Logistic_RFE', 'SVC_RFE', 'Random_RFE', 'DecisionTree_RFE'])
    return dataframe                      
        

dataset1=pd.read_csv("prep.csv",index_col=None)
df2=dataset1
df2 = pd.get_dummies(df2, drop_first=True)

indep_X=df2.drop('classification_yes', axis=1)
dep_Y=df2['classification_yes']

acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

rfelist=rfeFeature(indep_X , dep_Y , 3)

for i in rfelist:
    X_train,X_test,y_train,y_test=split_scaler(i,dep_Y)

    classifier,accuracy,report,X_test,y_test,cm=logistic(X_train,X_test,y_train,y_test)
    acclog.append(accuracy)

    classifier,accuracy,report,X_test,y_test,cm=svm_linear(X_train,X_test,y_train,y_test)  
    accsvml.append(accuracy)
    
    classifier,accuracy,report,X_test,y_test,cm=svm_NL(X_train,X_test,y_train,y_test)  
    accsvmnl.append(accuracy)
    
    classifier,accuracy,report,X_test,y_test,cm=knn(X_train,X_test,y_train,y_test)  
    accknn.append(accuracy)
    
    classifier,accuracy,report,X_test,y_test,cm=Navie(X_train,X_test,y_train,y_test)  
    accnav.append(accuracy)
    
    classifier,accuracy,report,X_test,y_test,cm=decision(X_train,X_test,y_train,y_test)  
    accdes.append(accuracy)
    
    classifier,accuracy,report,X_test,y_test,cm=random(X_train,X_test,y_train,y_test)  
    accrf.append(accuracy)
    
result=rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)





In [3]:
result

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
Logistic_RFE,0.94,0.94,0.94,0.94,0.94,0.94,0.94
SVC_RFE,0.87,0.87,0.87,0.87,0.87,0.87,0.87
Random_RFE,0.94,0.94,0.94,0.94,0.90,0.91,0.92
DecisionTree_RFE,0.98,0.98,0.98,0.98,0.79,0.97,0.97
